# **03 - Train Root Model (With Weighted Loss for Class Imbalance)**
Vision Transformer (ViT-B/16) untuk binary classification: Organic vs Inorganic

**Key Improvement:** Menggunakan weighted loss untuk handle class imbalance (Organic: 20%, Inorganic: 80%)

## **Import Library**

In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Paths
BASE_DIR = "../data/processed/level_1_root"
SAVED_MODELS_DIR = "../saved_models"
RESULTS_DIR = "../results"

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\n✓ Setup selesai!")

## **Hyperparameters**

In [ ]:
# Training Configuration
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

# Data paths
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "val")
TEST_DIR = os.path.join(BASE_DIR, "test")

print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Optimizer: SGD (momentum={MOMENTUM})")
print(f"Loss: CrossEntropyLoss dengan class weights")
print(f"Device: {device}")
print("="*60)

## **Data Augmentation & Loading**

In [ ]:
# Image transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

print("\n→ Loading datasets...")

# Create datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, data_transforms['train'])
val_dataset = datasets.ImageFolder(VAL_DIR, data_transforms['val'])
test_dataset = datasets.ImageFolder(TEST_DIR, data_transforms['test'])

# Create dataloaders
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4),
    'test': DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
}

# Class information
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"\nClass Names: {class_names}")
print(f"Number of Classes: {num_classes}")
print(f"\nDataset Sizes:")
print(f"  Train: {len(train_dataset)} images")
print(f"  Val: {len(val_dataset)} images")
print(f"  Test: {len(test_dataset)} images")

## **Calculate Class Weights (untuk handle imbalance)**

In [ ]:
# Extract labels dari training dataset
train_labels = [label for _, label in train_dataset.samples]
train_labels = np.array(train_labels)

# Count class distribution
class_counts = np.bincount(train_labels)
print("\n" + "="*60)
print("CLASS DISTRIBUTION (TRAINING SET)")
print("="*60)
for i, (class_name, count) in enumerate(zip(class_names, class_counts)):
    percentage = (count / len(train_labels)) * 100
    print(f"{class_name:15s}: {count:5d} images ({percentage:5.1f}%)")

# Calculate class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

print("\n" + "="*60)
print("CLASS WEIGHTS (untuk Loss Function)")
print("="*60)
for i, (class_name, weight) in enumerate(zip(class_names, class_weights)):
    print(f"{class_name:15s}: {weight:.4f}")

# Convert to tensor
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f"\nWeights Tensor: {class_weights_tensor}")

print("\n✓ Class weights calculated!")
print("  → Minority class (organic) gets higher weight")
print("  → Majority class (inorganic) gets lower weight")
print("  → Better recall untuk minority class")

## **Model Architecture (ViT-B/16)**

In [ ]:
print("\n→ Loading pre-trained Vision Transformer (ViT-B/16)...")

# Load pre-trained model
model_root = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

# Modify classification head untuk 2 classes
model_root.heads.head = nn.Linear(model_root.heads.head.in_features, num_classes)

# Move ke device
model_root = model_root.to(device)

print("\n✓ Model loaded!")
print(f"Total parameters: {sum(p.numel() for p in model_root.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model_root.parameters() if p.requires_grad):,}")

## **Loss Function & Optimizer (WITH WEIGHTED LOSS)**

In [ ]:
# 🎯 KEY: CrossEntropyLoss dengan class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer
optimizer = optim.SGD(
    model_root.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

print("\n" + "="*60)
print("LOSS FUNCTION & OPTIMIZER")
print("="*60)
print(f"Loss: CrossEntropyLoss with class_weights")
print(f"  → organic weight: {class_weights[0]:.4f}")
print(f"  → inorganic weight: {class_weights[1]:.4f}")
print(f"\nOptimizer: SGD")
print(f"  → Learning Rate: {LEARNING_RATE}")
print(f"  → Momentum: {MOMENTUM}")
print(f"  → Weight Decay: {WEIGHT_DECAY}")
print("="*60)

## **Training Function**

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    """
    Train model dengan tracking train/val metrics
    """
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    training_history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 50)
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_corrects = 0
        train_samples = 0
        
        for inputs, labels in dataloaders['train']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            with torch.set_grad_enabled(True):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                loss.backward()
                optimizer.step()
            
            # Statistics
            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_samples += inputs.size(0)
        
        train_loss_avg = train_loss / train_samples
        train_acc = train_corrects.double() / train_samples
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        val_samples = 0
        
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            with torch.set_grad_enabled(False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data)
            val_samples += inputs.size(0)
        
        val_loss_avg = val_loss / val_samples
        val_acc = val_corrects.double() / val_samples
        
        # Tracking
        training_history['train_loss'].append(train_loss_avg)
        training_history['train_acc'].append(train_acc.item())
        training_history['val_loss'].append(val_loss_avg)
        training_history['val_acc'].append(val_acc.item())
        
        print(f'Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc:.4f}')
        print(f'Val Loss: {val_loss_avg:.4f} | Val Acc: {val_acc:.4f}')
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f'→ Best model updated! (Acc: {best_acc:.4f})')
    
    # Load best weights
    model.load_state_dict(best_model_wts)
    
    time_elapsed = time.time() - since
    print(f'\n✓ Training complete!\n  Time: {int(time_elapsed)}s')
    
    return model, training_history

print("✓ Training function ready!")

## **Train Model**

In [ ]:
print("\n" + "="*60)
print("STARTING TRAINING...")
print("="*60)

model_root, training_history = train_model(
    model_root,
    criterion,
    optimizer,
    num_epochs=EPOCHS
)

print("\n✓ Training selesai!")

## **Save Best Model**

In [ ]:
model_save_path = os.path.join(SAVED_MODELS_DIR, 'root_model_best.pth')
torch.save(model_root.state_dict(), model_save_path)

print(f"✓ Model saved to: {model_save_path}")
print(f"  File size: {os.path.getsize(model_save_path) / 1e6:.1f} MB")

## **Visualize Training Curves**

In [ ]:
# Prepare data
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(epochs_range, training_history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, training_history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_title('Loss per Epoch', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs_range, training_history['train_acc'], 'b-', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, training_history['val_acc'], 'r-', label='Val Acc', linewidth=2)
axes[1].set_title('Accuracy per Epoch', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves_root_model.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training curves saved!")

## **Evaluate on Test Set**

In [ ]:
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

# Set model to eval mode
model_root.eval()

# Prediction
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        outputs = model_root(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate metrics
test_accuracy = np.mean(all_preds == all_labels)
print(f"\nTest Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print(f"\nConfusion Matrix:")
print(cm)

# Classification Report
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix - Test Set\n(Accuracy: {test_accuracy:.4f})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix_root_model.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Evaluation complete!")

## **Summary**

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nModel: Vision Transformer (ViT-B/16)")
print(f"Task: Binary Classification (Organic vs Inorganic)")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Optimizer: SGD (LR={LEARNING_RATE}, momentum={MOMENTUM})")
print(f"Loss: CrossEntropyLoss with class weights")
print(f"  → Organic weight: {class_weights[0]:.4f}")
print(f"  → Inorganic weight: {class_weights[1]:.4f}")

print(f"\nResults:")
print(f"  Final Train Accuracy: {training_history['train_acc'][-1]:.4f}")
print(f"  Final Val Accuracy: {training_history['val_acc'][-1]:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

print(f"\nOutput Files:")
print(f"  ✓ Model: {model_save_path}")
print(f"  ✓ Training Curves: {os.path.join(RESULTS_DIR, 'training_curves_root_model.png')}")
print(f"  ✓ Confusion Matrix: {os.path.join(RESULTS_DIR, 'confusion_matrix_root_model.png')}")
print("="*60)